Import Required Modules

In [94]:
from pyspark.sql import SparkSession
from pyspark.sql.functions  import col,when,avg,sum,count,desc,lit,when,row_number
from pyspark.sql.window import Window

Create Session

In [95]:
spark=SparkSession.builder.appName("pySpark Basics").getOrCreate()
print(f"Spark Session Created with Spark Version : {spark.version}")

Spark Session Created with Spark Version : 4.0.3


Create Random Data as DataFrame

In [96]:
employee_data=[
    (100,"Gourav","IT",10000,"Delhi"),
    (101,"Somya","HR",50000,"Bhopal"),
    (102,"Divya","Marketing",45000,"Delhi"),
    (103,"Rohan","Finance",40000,"Indore"),
    (104,"Dev","IT",35000,"Jaipur"),
    (105,"Amit","HR",20000,"Indore"),
    (106,"Bhavya","Finance",90000,"Bhopal"),
    (107,"Akshara","IT",85000,"Gwalior")

]
columns=["emp_id","emp_name","department","salary","city"]

df=spark.createDataFrame(
    data=employee_data,
    schema=columns
)

In [97]:
print("Original Schema\n")
df.show()
df.printSchema()

Original Schema

+------+--------+----------+------+-------+
|emp_id|emp_name|department|salary|   city|
+------+--------+----------+------+-------+
|   100|  Gourav|        IT| 10000|  Delhi|
|   101|   Somya|        HR| 50000| Bhopal|
|   102|   Divya| Marketing| 45000|  Delhi|
|   103|   Rohan|   Finance| 40000| Indore|
|   104|     Dev|        IT| 35000| Jaipur|
|   105|    Amit|        HR| 20000| Indore|
|   106|  Bhavya|   Finance| 90000| Bhopal|
|   107| Akshara|        IT| 85000|Gwalior|
+------+--------+----------+------+-------+

root
 |-- emp_id: long (nullable = true)
 |-- emp_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- city: string (nullable = true)



###DATA TRANSFORMATION
###1. Selecting & Renaming

In [98]:
df_selected=df.select(col('emp_id'),col('emp_name').alias("emp_Full_Name"),col('salary'))
df_selected.show()


+------+-------------+------+
|emp_id|emp_Full_Name|salary|
+------+-------------+------+
|   100|       Gourav| 10000|
|   101|        Somya| 50000|
|   102|        Divya| 45000|
|   103|        Rohan| 40000|
|   104|          Dev| 35000|
|   105|         Amit| 20000|
|   106|       Bhavya| 90000|
|   107|      Akshara| 85000|
+------+-------------+------+



In [99]:
# For only 3 Row data
df_selected.show(3)

+------+-------------+------+
|emp_id|emp_Full_Name|salary|
+------+-------------+------+
|   100|       Gourav| 10000|
|   101|        Somya| 50000|
|   102|        Divya| 45000|
+------+-------------+------+
only showing top 3 rows


###2. Filtering Rows

In [100]:
df_filtered=df.filter((col('department')=="IT") & (col('salary')>80000))
df_filtered.show()

+------+--------+----------+------+-------+
|emp_id|emp_name|department|salary|   city|
+------+--------+----------+------+-------+
|   107| Akshara|        IT| 85000|Gwalior|
+------+--------+----------+------+-------+



###3. Adding and Modifying Columns

In [101]:
df_transformed=df.withColumn("bonus",col("salary")*0.10).withColumn("salary_tier",when(col("salary")>=90000,lit("High")).otherwise(lit("Standard")))
df_transformed.show()

+------+--------+----------+------+-------+------+-----------+
|emp_id|emp_name|department|salary|   city| bonus|salary_tier|
+------+--------+----------+------+-------+------+-----------+
|   100|  Gourav|        IT| 10000|  Delhi|1000.0|   Standard|
|   101|   Somya|        HR| 50000| Bhopal|5000.0|   Standard|
|   102|   Divya| Marketing| 45000|  Delhi|4500.0|   Standard|
|   103|   Rohan|   Finance| 40000| Indore|4000.0|   Standard|
|   104|     Dev|        IT| 35000| Jaipur|3500.0|   Standard|
|   105|    Amit|        HR| 20000| Indore|2000.0|   Standard|
|   106|  Bhavya|   Finance| 90000| Bhopal|9000.0|       High|
|   107| Akshara|        IT| 85000|Gwalior|8500.0|   Standard|
+------+--------+----------+------+-------+------+-----------+



In [102]:
df_transformed.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(salary_tier, CASE WHEN '`>=`('salary, 90000) THEN High ELSE Standard END, None)]
+- Project [emp_id#707L, emp_name#708, department#709, salary#710L, city#711, (cast(salary#710L as double) * 0.1) AS bonus#765]
   +- LogicalRDD [emp_id#707L, emp_name#708, department#709, salary#710L, city#711], false

== Analyzed Logical Plan ==
emp_id: bigint, emp_name: string, department: string, salary: bigint, city: string, bonus: double, salary_tier: string
Project [emp_id#707L, emp_name#708, department#709, salary#710L, city#711, bonus#765, CASE WHEN (salary#710L >= cast(90000 as bigint)) THEN High ELSE Standard END AS salary_tier#766]
+- Project [emp_id#707L, emp_name#708, department#709, salary#710L, city#711, (cast(salary#710L as double) * 0.1) AS bonus#765]
   +- LogicalRDD [emp_id#707L, emp_name#708, department#709, salary#710L, city#711], false

== Optimized Logical Plan ==
Project [emp_id#707L, emp_name#708, department#709, salary

###4. Aggregation & Grouping

In [103]:

df_agg = (
    df.groupBy("department")
      .agg(
          avg("salary").alias("avg_salary"),   # use consistent alias
          count("emp_id").alias("emp_count")
      )
      .orderBy(desc("avg_salary"))             # now matches alias
)

df_agg.show()

+----------+------------------+---------+
|department|        avg_salary|emp_count|
+----------+------------------+---------+
|   Finance|           65000.0|        2|
| Marketing|           45000.0|        1|
|        IT|43333.333333333336|        3|
|        HR|           35000.0|        2|
+----------+------------------+---------+



###5. Window Function


ranking employees by salary within each department

In [104]:

window_spec= Window.partitionBy('department').orderBy(desc("salary"))
df_ranked=df.withColumn("Salary_rank",row_number().over(window_spec))
df_ranked.show()

+------+--------+----------+------+-------+-----------+
|emp_id|emp_name|department|salary|   city|Salary_rank|
+------+--------+----------+------+-------+-----------+
|   106|  Bhavya|   Finance| 90000| Bhopal|          1|
|   103|   Rohan|   Finance| 40000| Indore|          2|
|   101|   Somya|        HR| 50000| Bhopal|          1|
|   105|    Amit|        HR| 20000| Indore|          2|
|   107| Akshara|        IT| 85000|Gwalior|          1|
|   104|     Dev|        IT| 35000| Jaipur|          2|
|   100|  Gourav|        IT| 10000|  Delhi|          3|
|   102|   Divya| Marketing| 45000|  Delhi|          1|
+------+--------+----------+------+-------+-----------+



###Stoping the Spark Session

In [105]:
spark.stop()
print("Spark Session has stopped ")

Spark Session has stopped 


##Directed Acyclic Graph (DAG)

###Step 1.Initialization and Setup

In [106]:
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col,lit

In [107]:
spark=SparkSession.builder.appName("DAG session").getOrCreate()
print("Spark Session has been created succesfully")

Spark Session has been created succesfully


###Lazy Evaluation

###Create a large range of numbers (10 million rowss)

In [108]:
start_time=time.time()
df_range=spark.range(0,10000000)
print(f"Time to define range:{time.time()-start_time: .4f} seconds (Almost Instantaneous)")

Time to define range: 0.0415 seconds (Almost Instantaneous)


Add an expensive mathematical Transformation

In [109]:
df_transform=df_range.withColumn("Multiplied_val",
                                 col("id")*3).filter(col("Multiplied_val")%2==0)
print(f"Time to define Transformations: {time.time()- start_time:.4f} seconds (Still no work is done)")

Time to define Transformations: 0.1229 seconds (Still no work is done)


###Viewing The logical and Physical DAG

In [110]:
df_transform.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('`%`('Multiplied_val, 2), 0)
+- Project [id#838L, (id#838L * cast(3 as bigint)) AS Multiplied_val#839L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Analyzed Logical Plan ==
id: bigint, Multiplied_val: bigint
Filter ((Multiplied_val#839L % cast(2 as bigint)) = cast(0 as bigint))
+- Project [id#838L, (id#838L * cast(3 as bigint)) AS Multiplied_val#839L]
   +- Range (0, 10000000, step=1, splits=Some(2))

== Optimized Logical Plan ==
Project [id#838L, (id#838L * 3) AS Multiplied_val#839L]
+- Filter (((id#838L * 3) % 2) = 0)
   +- Range (0, 10000000, step=1, splits=Some(2))

== Physical Plan ==
*(1) Project [id#838L, (id#838L * 3) AS Multiplied_val#839L]
+- *(1) Filter (((id#838L * 3) % 2) = 0)
   +- *(1) Range (0, 10000000, step=1, splits=2)



###Triger The Action

In [111]:
action_start_time=time.time()
total_rows=df_transform.count()
action_end_time=time.time()

print(f"Resulting Row Count: {total_rows}")
print(f"Time taken to Execute Action:{action_end_time-action_start_time: .2f}seconds")



Resulting Row Count: 5000000
Time taken to Execute Action: 0.58seconds


###Performing Optimization


In [112]:
### Re-run the whole action without Cache
start_time=time.time()
df_transform.count()
print(f"Time for 2nd run (No cahche): {time.time()-start_time: .2f} seconds")

Time for 2nd run (No cahche):  0.57 seconds


In [113]:
## Re-run with cache
## .cache() is a lazy transformation that tells Spark to store the result in memory once computed
df_transform.cache()
df_transform.count()
start_time=time.time()
df_transform.count()
print(f"Time for 3rd run (cached): {time.time()-start_time: .2f} seconds")

Time for 3rd run (cached):  0.24 seconds
